# Documentation Opérationnelle
## Projet ML - Segmentation et Scoring Client Bancaire

**Version**: 2.0 
**Date**: Septembre 2025  
**Équipe**: Data Science - Crédit Agricole

---

## Table des Matières

1. [Vue d'ensemble du projet](#vue-densemble-du-projet)
2. [Architecture système](#architecture-système)
3. [Modules principaux](#modules-principaux)
4. [Workflows opérationnels](#workflows-opérationnels)
5. [Configuration et paramétrage](#configuration-et-paramétrage)
6. [Guide d'utilisation](#guide-dutilisation)
7. [Monitoring et maintenance](#monitoring-et-maintenance)
8. [Gestion des modèles](#gestion-des-modèles)
9. [Troubleshooting](#troubleshooting)

---

## Vue d'ensemble du projet

### Objectif métier
Système de segmentation client automatisée basée sur le clustering stratifié par âge et le scoring PNB intra-cluster, permettant l'identification des potentiels commerciaux et la priorisation des actions.

### Périmètre fonctionnel
- **Extraction** : Données socio-démographiques, organisationnelles et financières depuis l'entrepôt de données
- **Clustering** : Segmentation par strates d'âge avec k-means adaptatif
- **Scoring** : Calcul de scores PNB pondérés et potentiels par cluster
- **Segmentation finale** : Classification combinée (score PNB + note MIRE)

### Indicateurs clés
- **Volume traité** : ~310k clients
- **Clusters créés** : ~50 segments
- **Qualité clustering** : Silhouette > 0.65
- **Potentiel identifié** : ~135M€
- **Couverture** : 100% clients clusterisés

---

## Architecture système

### Principe architectural
Architecture modulaire avec pipeline centralisée, configuration unifiée et gestion intelligente des modèles.

```
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│  DataExtractor  │───▶│DimensionOptimizer│───▶│  AgeClusterer   │
└─────────────────┘    └─────────────────┘    └─────────────────┘
         │                       │                       │
         ▼                       ▼                       ▼
┌─────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│     Cache       │    │   Config        │    │ ClientScoring   │
└─────────────────┘    └─────────────────┘    └─────────────────┘
```

### Structure des répertoires
```
SEGPOTDYN/
├── src/
│   ├── __init__.py              # Point d'entrée centralisé
│   ├── config.py                # Configuration unifiée
│   ├── data_extractor.py        # Extraction données
│   ├── dimension_optimizer.py   # Optimisation dimensions
│   ├── age_clustering.py        # Clustering stratifié
│   ├── client_scoring.py        # Scoring clients
│   └── advanced_pipeline.py     # Pipeline avancé
├── models/                      # Modèles persistés
├── cache/                       # Cache données
└── logs/                        # Logs système
```

---

## Modules principaux

### 1. Configuration (`config.py`)
**Responsabilité** : Configuration centralisée et système de logging unifié

**Composants clés** :
- `Config` : Configuration principale
- `LoggerConfig` : Gestion des logs avec rotation
- `ProjectConfig` : Chemins et paramètres projet

**Paramètres critiques** :
```python
CLUSTERING = {
    'age_bins': 5,              # Taille strates (ans)
    'k_range': (2, 6),          # Plage K pour k-means
    'min_cluster_size': 800,    # Taille minimale cluster
    'min_strata_size': 10000    # Taille minimale strate
}
```

### 2. Extraction de données (`data_extractor.py`)
**Responsabilité** : Extraction unifiée depuis l'entrepôt Spark

**Fonctionnalités** :
- Requêtes SQL optimisées par domaine (socle, organisationnel, financier)
- Système de cache intelligent
- Gestion des erreurs et logs détaillés

**Requêtes principales** :
- `core` : Données socio-démographiques avec filtres âge
- `financial_pnb` : PNB par univers (COLL, CRED, SERV, ASSU)
- `financial_patrimoine` : Épargne et DAV
- `organizational` : Hiérarchie commerciale

### 3. Optimisation dimensionnelle (`dimension_optimizer.py`)
**Responsabilité** : Transformation des variables pour clustering

**Modes disponibles** :
- **Global** : 11 dimensions avec tranches d'âge encodées
- **Stratifié** : 8 dimensions + âge numérique

**Variables créées** :
1. `Situation_Couple/Celibataire` : État matrimonial binaire
2. `AGE` : Numérique (mode stratifié) ou tranches (mode global)
3. `REVENU_Score` : Score ordinal (Frontalier=3, CSP+, CSP moyen, CSP-)
4. `Enfants_Avec/Sans` : Présence enfants binaire
5. `Logement_Proprietaire/NonProprietaire` : Statut logement binaire

### 4. Clustering stratifié (`age_clustering.py`)
**Responsabilité** : Clustering par strates d'âge avec persistance modèles

**Stratégie de clustering** :
- Strates fixes de 5 ans : 18-22, 23-27, ..., 63+
- K-means adaptatif par strate (K=2 à 6)
- Contraintes métier : min 800 clients/cluster, min 10k/strate
- Sauvegarde automatique des modèles avec métadonnées

**Métriques de qualité** :
- Score silhouette par strate et global
- Distribution des clusters
- Importance des features
- Historique des performances

### 5. Scoring clients (`client_scoring.py`)
**Responsabilité** : Calcul scores PNB et potentiels intra-cluster

**Méthodologie CA** :
1. **Pondération** : Moyennes absolues PNB par cluster
2. **Normalisation** : Min-max par cluster
3. **Score client** : Somme pondérée des PNB normalisés
4. **Potentiels** : Écart aux top 25% par cluster
5. **Segmentation finale** : Score PNB + Note MIRE

**Segments finaux** :
- **A entretenir** (8.4%) : Score combiné = 8
- **A développer** (33.7%) : Score combiné ≥ 6
- **A stimuler** (31.3%) : Score combiné ≥ 4
- **A construire** (26.6%) : Score combiné < 4

### 6. Pipeline avancé (`advanced_pipeline.py`)
**Responsabilité** : Orchestration avec gestion intelligente des modèles

**Logique de réutilisation** :
- **Par défaut** : Réutilise modèles existants
- **Recalcul automatique** : Si aucun modèle ou erreur chargement
- **Force retrain** : Paramètre explicite pour recalcul
- **Monitoring détaillé** : Métriques de dérive et alertes

---

## Workflows opérationnels

### Workflow principal - Mode production

```mermaid
graph TD
    A[Démarrage Pipeline] --> B{Modèle existant?}
    B -->|Oui| C[Chargement modèle]
    B -->|Non| D[Premier entraînement]
    C --> E[Analyse métriques détaillées]
    E --> F[Logging signaux dérive]
    F --> G[Application clustering]
    D --> H[Entraînement modèle]
    H --> I[Sauvegarde modèle]
    I --> G
    G --> J[Scoring clients]
    J --> K[Segmentation finale]
    K --> L[Persistance résultats]
```

### Workflow de développement

```python
from src.advanced_pipeline import run_advanced_pipeline

# Exécution normale (réutilise modèle)
results = run_advanced_pipeline(
    mode='dev',
    clustering_mode='stratifie',
    sample_size=10000,
    force_retrain=False
)

# Force le recalcul si nécessaire
results_retrain = run_advanced_pipeline(
    mode='dev',
    clustering_mode='stratifie',
    force_retrain=True
)
```

### Workflow de monitoring

```python
from src.advanced_pipeline import get_pipeline_model_status

# Vérification statut modèles
status = get_pipeline_model_status()
print(f"Qualité modèle: {status.get('avg_silhouette', 'N/A')}")
print(f"Âge modèle: {status.get('training_date', 'N/A')}")
```

---

## Configuration et paramétrage

### Paramètres critiques à surveiller

#### Clustering
```python
CLUSTERING = {
    'age_bins': 5,              # Ajuster selon distribution âges
    'k_range': (2, 6),          # Élargir si silhouette faible
    'min_cluster_size': 800,    # Réduire si trop de strates ignorées
    'min_strata_size': 10000    # Adapter selon volume données
}
```

#### Extraction
```python
EXTRACTION = {
    'age_filters': {'min': 18, 'max': 100},
    'nb_majeur_min': 1,         # Filtre base données
    'default_sample_size': 10000
}
```

#### Scoring
```python
SCORING = {
    'top_client_pct': 0.25,     # Percentile pour calcul potentiels
    'pnb_columns': ['PNB_COLL', 'PNB_CRED', 'PNB_SERV', 'PNB_ASSU']
}
```

### Variables d'environnement
```bash
DATABASE_NAME=DEFAULT_DB      # Base de données source
DEBUG_IMPORTS=true           # Debug imports au démarrage
```

---

## Guide d'utilisation

### Utilisation de base

```python
# Import du pipeline principal
from src.advanced_pipeline import AdvancedBankingPipeline

# Initialisation
pipeline = AdvancedBankingPipeline(
    mode='prod',                    # prod, dev, test
    clustering_mode='stratifie',    # global, stratifie
    force_retrain=False
)

# Exécution
results = pipeline.run()

# Accès aux résultats
df_scored = results['data']           # DataFrame final
metrics = results['metrics']         # Métriques qualité
metadata = results['metadata']       # Métadonnées exécution
```

### Utilisation par composants

```python
# Extraction seule
from src.data_extractor import DataExtractor
extractor = DataExtractor(use_cache=True)
df_raw = extractor.extract_all_data()

# Clustering seul
from src.age_clustering import AgeClusterer
clusterer = AgeClusterer()
df_clustered = clusterer.fit_transform(df_processed)

# Scoring seul
from src.client_scoring import ClientScoringEngine
scorer = ClientScoringEngine()
df_scored = scorer.calculate_scores(df_clustered, df_financial)
```

### Modes d'exécution

#### Mode Development
- Sauvegarde fichiers CSV locaux
- Cache activé par défaut
- Logs détaillés
- Échantillonnage possible

#### Mode Production
- Persistance en base Spark
- Cache désactivé (données fraîches)
- Logs optimisés
- Volume complet

#### Mode Test
- Échantillonnage forcé
- Validation rapide
- Métriques allégées

---

## Gestion des modèles

### Cycle de vie des modèles

#### Création
- Entraînement automatique si aucun modèle
- Sauvegarde avec timestamp et métadonnées
- Enregistrement historique performances

#### Réutilisation (par défaut)
- Chargement modèle le plus récent
- Validation compatibilité données
- Monitoring qualité application

#### Recalcul (décisionnel)
- Sur dégradation détectée
- Sur dérive démographique importante
- Sur demande explicite (force_retrain=True)

### Structure modèles persistés

```python
model_data = {
    'model_version': '20250919_140449',
    'age_strata': {...},                    # Définitions strates
    'scaler_models': {...},                 # Normaliseurs par strate
    'kmeans_models': {...},                 # Modèles k-means par strate
    'quality_metrics': {...},               # Métriques qualité détaillées
    'training_metadata': {...},             # Contexte entraînement
    'config': {...}                         # Configuration utilisée
}
```

## Troubleshooting

### Problèmes possibles

#### Échec extraction données
**Symptômes** : DataFrame vide, erreurs SQL
**Causes** : 
- Base de données indisponible
- Requêtes SQL incompatibles
- Filtres trop restrictifs

**Solutions** :
```python
# Vérification connexion
extractor = DataExtractor(use_cache=False)
df_sample = extractor.get_sample(100)

# Test requête basique
sql_simple = "SELECT COUNT(*) FROM {}.connaissance_client".format(Config.DATABASE)
```

#### Clustering échoue
**Symptômes** : Strates ignorées, silhouette faible
**Causes** :
- Strates sous-représentées
- Paramètres k_range inadaptés
- Données déséquilibrées

**Solutions** :
```python
# Ajustement paramètres
CLUSTERING['min_strata_size'] = 5000  # Réduire seuil
CLUSTERING['k_range'] = (2, 8)        # Élargir plage K
```

#### Modèles incompatibles
**Symptômes** : Erreur chargement, features manquantes
**Causes** :
- Changement structure données
- Version modèle obsolète
- Corruption fichier modèle

**Solutions** :
```python
# Force nouveau modèle
pipeline = AdvancedBankingPipeline(force_retrain=True)

# Nettoyage cache
extractor.clear_cache()
```

### Logs de diagnostic

#### Localisation logs
```bash
/path/to/project/logs/
├── advanced_pipeline_YYYYMMDD.log
├── data_extractor_YYYYMMDD.log
├── age_clustering_YYYYMMDD.log
└── client_scoring_YYYYMMDD.log
```

### Procédures de récupération

#### Récupération après panne
1. Vérifier intégrité fichiers cache et modèles
2. Nettoyer logs corrompus si nécessaire
3. Relancer avec force_retrain si modèles suspects

#### Reset complet
```python
# Nettoyage complet
from src.data_extractor import DataExtractor
extractor = DataExtractor()
extractor.clear_cache()

# Suppression modèles
import shutil
shutil.rmtree('/path/to/models/', ignore_errors=True)

# Relance complète
from src.advanced_pipeline import run_advanced_pipeline
results = run_advanced_pipeline(mode='dev', force_retrain=True)
```

---

*Dernière mise à jour du document: Septembre 2025*